# 04 - Pipeline ranking and dashboard

Crossing the two signals.

| | **intent stated** | **intent absent** |
|---|---|---|
| **high pressure** | Act now | Watch |
| **low pressure** | Opportunistic | Deprioritise |

A bank under heavy pressure that has said nothing is a different call from one
that has announced a programme, and both differ from a comfortable bank being
opportunistic.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)


In [2]:
from src import ranking as R, pressure as P, viz as V

ranked  = R.build_ranking()
summary = P.country_summary(ranked)
print(R.coverage_note(ranked))
R.pipeline_table(ranked, 15).round(1)

4 of 48 banks carry a provenance-verified disclosure quote. The remaining 44 are ranked on structured data only and are reported as intent-unconfirmed rather than as silent.


,rank,bank,country,loans_npl,npl_ratio,stage2_ratio,coverage_ratio,net_npl_to_cet1,pressure_score,quadrant
0,1,Deutsche Pfandbriefbank AG,DE,2021.9,5.6,15.2,35.3,48.4,86.9,Act now
1,2,Landesbank Hessen-Thüringen Girozentrale,DE,3919.0,3.1,15.2,29.6,29.9,85.8,Act now (intent unconfirmed)
2,3,Groupe BPCE,FR,26279.3,2.6,13.7,39.4,21.6,83.8,Act now (intent unconfirmed)
3,4,"Banco Santander, S.A.",ES,35924.5,3.3,6.8,39.6,26.7,83.8,Act now (intent unconfirmed)
4,5,ATLANTIC LUX HOLDCO S.A R.L.,DE,1477.0,4.4,13.5,21.5,39.7,83.5,Act now (intent unconfirmed)
5,6,Banca Monte dei Paschi di Siena S.p.A.,IT,3064.4,3.6,11.3,46.0,18.4,80.0,Watch
6,7,Bpifrance,FR,3932.8,6.7,17.9,23.0,12.8,79.2,Act now (intent unconfirmed)
7,8,Société générale S.A.,FR,14006.9,2.8,8.2,44.0,14.9,74.2,Act now (intent unconfirmed)
8,9,Confédération Nationale du Crédit Mutuel,FR,20134.1,2.7,7.5,46.3,14.8,71.5,Act now (intent unconfirmed)
9,10,BNP Paribas,FR,26870.4,2.8,6.8,48.5,14.1,71.0,Act now (intent unconfirmed)


## Why each name is on the list

The ranking is only useful if it can be argued with. Each row carries the
binding constraint in plain language, so a reader who disagrees can see exactly
which input to push back on.

In [3]:
for _, r in ranked.head(8).iterrows():
    print(f"{r['rank']:>2}. {r['bank'][:46]:<46} {r['quadrant']}")
    print(f"    {r['why']}\n")

 1. Deutsche Pfandbriefbank AG                     Act now
    NPL ratio 5.6%; Stage 2 at 15.2% signals further formation; coverage only 35% so a sale crystallises loss; unreserved NPLs equal 48% of CET1.

 2. Landesbank Hessen-Thüringen Girozentrale       Act now (intent unconfirmed)
    NPL ratio 3.1%; Stage 2 at 15.2% signals further formation; coverage only 30% so a sale crystallises loss; unreserved NPLs equal 30% of CET1.

 3. Groupe BPCE                                    Act now (intent unconfirmed)
    NPL ratio 2.6%; Stage 2 at 13.7% signals further formation; coverage only 39% so a sale crystallises loss.

 4. Banco Santander, S.A.                          Act now (intent unconfirmed)
    NPL ratio 3.3%; coverage only 40% so a sale crystallises loss; unreserved NPLs equal 27% of CET1.

 5. ATLANTIC LUX HOLDCO S.A R.L.                   Act now (intent unconfirmed)
    NPL ratio 4.4%; Stage 2 at 13.5% signals further formation; coverage only 22% so a sale crystallises loss; u

In [4]:
ranked['quadrant'].value_counts()

quadrant
Deprioritise                    33
Act now (intent unconfirmed)    11
Act now                          2
Watch                            2
Name: count, dtype: int64

## Figures and dashboard

In [5]:
V.fig_pressure_map(ranked)

In [6]:
V.fig_country_comparison(summary)

In [7]:
import pathlib
root = pathlib.Path.cwd().parent
path = V.build_dashboard(ranked, summary)
V.export_figures(ranked, summary)
print("Dashboard written to:", pathlib.Path(path).relative_to(root))

Dashboard written to: docs/dashboard.html


## Known limits

- **UK is out of scope.** UK banks left the EBA sample after Brexit, so one leg
  of the mandate is not addressable from this source.
- **The intent layer is not yet live.** See notebook 03 -- the pipeline runs but
  the corpus needs verified quotes before it produces signal.
- **The CET1 requirement is a proxy.** Banks disclose institution-specific SREP
  requirements; a single 10.5% floor is used here as a transparent
  approximation, set in `src/config.py`.
- **Percentile scoring is relative.** If every bank in the universe is
  comfortable, someone still ranks first. The score ranks within this universe;
  it is not an absolute distress measure.